# **7일차 실습: 도메인 특화 도구 만들기 및 테스트**

## 학습 목표
1. LangChain Tool 개념 이해
2. AI를 활용한 도구 자동 생성
3. 생성한 도구 테스트
4. 도메인 특화 에이전트에 적용

## 실습 단계
1. **도구 설계**: 팀 프로젝트에 필요한 도구 정의
2. **도구 생성**: AI를 활용하여 도구 코드 자동 생성
3. **도구 테스트**: 생성된 도구가 올바르게 동작하는지 확인
4. **에이전트 적용**: domain-agent에 통합 (다음 단계)

---

## 1. AI 도구 생성 프롬프트 확인

팀 프로젝트에 필요한 도구를 AI로 자동 생성하기 위한 프롬프트입니다.

**프롬프트 파일 위치**: `../../make_tool_prompt.txt`

In [2]:
# 프롬프트 파일 읽기
with open("../make_tool_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()

print("=" * 80)
print("AI 도구 생성 프롬프트")
print("=" * 80)
print(prompt_template)
print("\n✓ 프롬프트 로드 완료")
print("\n💡 이 프롬프트를 ChatGPT, Claude 등에 복사하여 사용하세요!")

AI 도구 생성 프롬프트
당신은 LangChain/LangGraph 기반 AI Agent를 위한 Python Tool 개발 전문가입니다.

아래 **[사용자 요구사항]** 에 작성된 내용을 바탕으로 LangChain Tool을 생성하세요.

---

# 사용자 요구사항

## 여기에 원하는 기능을 작성하세요.

<이곳에 원하는 Tool 기능을 작성>

---

# 구현 규칙

다음 규칙을 반드시 지키세요.

## 출력 형식

* Python 코드만 출력합니다.
* 코드 외의 설명은 출력하지 않습니다.
* `from langchain_core.tools import tool`을 사용합니다.
* `@tool(parse_docstring=True)` 데코레이터를 사용합니다.
* Python 3.11 이상 기준으로 작성합니다.

## 함수 작성 규칙

* 함수명은 기능에 맞게 작성합니다.
* 모든 매개변수에는 타입 힌트를 작성합니다.
* 반환 타입도 작성합니다.
* 필요한 import는 함수 내부에서 수행합니다.
* 가능한 표준 라이브러리를 우선 사용합니다.
* 외부 라이브러리가 필요한 경우 import도 함께 작성합니다.

## Docstring

반드시 Google Style Docstring을 작성합니다.

예시 형식

```python
"""도구 설명

Args:
    parameter1: 설명
    parameter2: 설명

Returns:
    반환값 설명
"""
```

## 예외 처리

반드시 예외 처리를 구현합니다.

형식

```python
try:
    ...
    return "성공 메시지"
except Exception as e:
    return f"실패: {str(e)}"
```

## 테스트 코드

마지막에 아래 코드를 추가합니다.

```python
print(f"도구 이름: {함수명.name}")
print(f"도구 설명: {함수명.description}")
```

## 코드 품질

* 읽기 쉬운 코드로 작성합니다.
* 

## 2. 도구 설계 가이드

### 좋은 도구의 조건

1. **단일 책임**: 하나의 명확한 기능만 수행
2. **명확한 입력/출력**: 매개변수와 반환값이 명확
3. **에러 처리**: 예외 상황을 적절히 처리
4. **좋은 설명**: Docstring으로 도구의 기능을 명확히 설명

### 도메인별 도구 예시

**쇼핑 도메인:**
- 상품 검색
- 가격 비교
- 재고 확인
- 리뷰 조회

**법령 도메인:**
- 법령 검색
- 조문 조회
- 판례 검색
- 법령 해석

**의료 도메인:**
- 증상 검색
- 병원 찾기
- 약 정보 조회
- 건강 정보 제공

**여행 도메인:**
- 항공권 검색
- 호텔 검색
- 관광지 정보
- 날씨 확인

---

## 3. 도구 생성 프로세스

### Step 1: 팀 프로젝트 도메인 및 필요한 도구 정의

**팀 도메인:** 의심스러운 이메일을 분석하고 피싱 여부를 판단하는 AI Agent

**필요한 도구 목록:**
1. EmailParserTool: 이메일 원문에서 발신자, 수신자, 제목, 본문, URL, 첨부파일, 의심 키워드를 추출
2. URLSecurityCheckTool: 이메일 내 URL의 위험도를 검사
3. DomainLookupTool: 발신자 도메인의 형식과 의심 패턴을 분석
4. ThreatIntelligenceSearchTool: 과거 피싱 사례 및 위협 인텔리전스 연관성을 확인
5. RiskScoreTool: 모든 정보를 종합하여 최종 판정 수행


```
팀 도메인: 의심스러운 이메일을 분석하고 피싱 여부를 판단하는 AI Agent

필요한 도구 목록:
1. 도구명: EmailParserTool [이메일 원문에서 헤더, 본문, URL, 이메일 주소 등 핵심 정보를 추출하는 도구]
   - 입력: 이메일 원문, 발신자, 제목, 본문, 이메일 헤더
   - 출력: 발신자, 수신자, 제목, 날짜, 추출된 URL, 추출된 이메일 주소 요약
   - 역할: 이메일 내용을 구조적으로 분해하여 후속 분석에 필요한 정보를 추출

2. 도구명: URLSecurityCheckTool [이메일에 포함된 URL의 위험도를 간단히 검사하는 도구]
   - 입력: 이메일에서 추출된 URL
   - 출력: URL 판정 결과, 위험 점수, 의심 신호
   - 역할: 단축 URL, 비정상 스킴, 과도한 하이픈, 실행 파일 경로 등 위험한 링크를 탐지

3. 도구명: DomainLookupTool [발신자 도메인의 형식, 신뢰도, 의심 신호를 분석하는 도구]
   - 입력: 도메인 문자열 또는 발신자 이메일 주소
   - 출력: 도메인 분석 결과, 위험 점수, 의심 신호
   - 역할: 발신자 도메인이 정상적인지, 피싱에 자주 사용되는 패턴이 있는지 판단

4. 도구명: ThreatIntelligenceSearchTool [과거 피싱 사례, IOC, 악성 도메인/URL 관련 위협 인텔리전스 정보를 확인하는 도구]
   - 입력: 검색어, 도메인, URL, IOC 관련 문자열
   - 출력: 매칭된 위협 키워드, 참고 위험도, 검색 요약
   - 역할: 과거 피싱 사례나 알려진 악성 요소와 현재 이메일의 연관성을 확인

5. 도구명: RiskScoreTool [이메일 전체 요소를 종합해 최종 위험도를 산정하고 안전/의심/악성으로 판정하는 도구]
   - 입력: 이메일 본문, URL 목록, 발신자 이메일, 발신자 도메인, 첨부파일 정보, 과거 위협 정보
   - 출력: 최종 판정, 위험 점수, 감지된 신호 목록
   - 역할: 이메일 분석 결과를 종합하여 최종 피싱 위험도를 결정
```

### Step 2: AI로 도구 생성하기

**사용 방법:**

1. 위의 프롬프트 템플릿을 복사
2. `<이곳에 원하는 Tool 기능을 작성>` 부분에 팀의 도구 요구사항 작성
3. ChatGPT, Claude 등에 입력하여 코드 생성
4. 생성된 코드를 아래 셀에 붙여넣기

**예시 입력:**
```
쇼핑 도메인의 상품 검색 도구를 만들어주세요.

기능:
- 상품명으로 검색
- 가격 범위 필터링
- 카테고리 필터링
- 검색 결과를 JSON 형태로 반환
```

---

## 4. 생성된 도구 코드 테스트

`tools.py`에 생성된 도구 코드를 아래 셀에 붙여넣어 실행할 수 있습니다.

**중요:**
- 코드를 실행하기 전에 반드시 검토하세요
- 필요한 외부 라이브러리가 있다면 먼저 설치하세요
- 실제 API 키가 필요한 경우 .env 파일에 추가하세요


In [ ]:
from langchain_core.tools import tool
from email import policy
from email.parser import Parser
from urllib.parse import urlparse
import re


@tool(parse_docstring=True)
def EmailParserTool(email_text: str) -> str:
    """이메일 원문에서 피싱 분석에 필요한 정보를 추출합니다.

    Args:
        email_text: 분석할 이메일 원문입니다.

    Returns:
        발신자, 수신자, 제목, 본문, URL, 첨부파일 및 피싱 의심 키워드 분석 결과입니다.
    """
    try:
        msg = Parser(policy=policy.default).parsestr(email_text)
        sender = msg.get("From", "정보 없음")
        recipient = msg.get("To", "정보 없음")
        subject = msg.get("Subject", "정보 없음")

        body_parts = []
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        content = part.get_content()
                        if "\\u" in content:
                            try:
                                content = content.encode("utf-8").decode("unicode_escape")
                            except Exception:
                                pass
                        if content:
                            body_parts.append(content)
                    except Exception:
                        pass
        else:
            try:
                body = msg.get_content()
                if "\\u" in body:
                    try:
                        body = body.encode("utf-8").decode("unicode_escape")
                    except Exception:
                        pass
                if body:
                    body_parts.append(body)
            except Exception:
                pass

        body = "\n".join(body_parts).strip() or email_text
        urls = re.findall(r'''https?://[^\s<>"'\)\]]+''', body)
        attachments = []
        for part in msg.iter_attachments():
            filename = part.get_filename()
            if filename:
                attachments.append(filename)

        suspicious_keywords = [
            "긴급", "즉시", "계정 정지", "계정이 잠겼", "비밀번호", "인증", "로그인", "결제", "송금",
            "보안 경고", "urgent", "verify", "verification", "password", "account suspended", "login",
            "payment", "security alert"
        ]
        search_text = f"{subject}\n{body}".lower()
        found_keywords = [keyword for keyword in suspicious_keywords if keyword.lower() in search_text]

        result = f"""
[EmailParserTool 분석 결과]

발신자: {sender}
수신자: {recipient}
제목: {subject}

본문:
{body[:3000]}

추출된 URL:
{chr(10).join(urls) if urls else "없음"}

첨부파일:
{chr(10).join(attachments) if attachments else "없음"}

피싱 의심 키워드:
{", ".join(found_keywords) if found_keywords else "탐지되지 않음"}

URL 개수: {len(urls)}
첨부파일 개수: {len(attachments)}
"""
        return result.strip()
    except Exception as e:
        return f"EmailParserTool 실행 실패: {str(e)}"


@tool(parse_docstring=True)
def URLSecurityCheckTool(url: str) -> str:
    """URL의 위험도를 간단히 검사합니다.

    Args:
        url: 검사할 URL 문자열입니다.

    Returns:
        URL 위험도 검사 결과 문자열입니다.
    """
    try:
        target = url.strip()
        if not target:
            return "실패: URL이 비어 있습니다."

        parsed = urlparse(target if target.startswith(("http://", "https://")) else f"http://{target}")
        host = (parsed.netloc or parsed.path).lower()

        score = 0
        signals = []
        if parsed.scheme not in {"http", "https"}:
            score += 10
            signals.append("비정상적인 스킴")
        if "@" in host:
            score += 20
            signals.append("@ 포함 URL")
        if len(host) > 30:
            score += 5
            signals.append("과도하게 긴 호스트명")
        if host.count("-") >= 3:
            score += 5
            signals.append("하이픈 과다 사용")

        suspicious_domains = {"bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly"}
        if any(host.endswith(domain) or domain in host for domain in suspicious_domains):
            score += 20
            signals.append("단축 URL 또는 위험 도메인")
        if re.search(r"[\u4e00-\u9fff]", host):
            score += 10
            signals.append("유니코드/한자 도메인 의심")
        if any(ext in target.lower() for ext in [".exe", ".scr", ".bat", ".cmd", ".js", ".vbs"]):
            score += 15
            signals.append("실행 파일형 경로 포함")

        verdict = "안전" if score < 15 else "의심" if score < 35 else "악성"
        return "\n".join([
            f"URL: {target}",
            f"판정: {verdict}",
            f"위험 점수: {score}/100",
            f"감지 신호: {', '.join(signals) if signals else '없음'}",
        ])
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def DomainLookupTool(domain: str) -> str:
    """발신자 도메인의 기본 특성을 분석합니다.

    Args:
        domain: 분석할 도메인 문자열입니다.

    Returns:
        도메인 분석 결과 문자열입니다.
    """
    try:
        value = domain.strip().lower()
        if not value:
            return "실패: 도메인이 비어 있습니다."
        if "@" in value:
            value = value.split("@", 1)[1]

        score = 0
        signals = []
        if not re.match(r"^[a-z0-9.-]+\.[a-z]{2,}$", value):
            score += 20
            signals.append("도메인 형식 비정상")
        if value.startswith("xn--") or ".xn--" in value:
            score += 10
            signals.append("Punycode 사용")
        if value.count(".") >= 3:
            score += 5
            signals.append("서브도메인 다수")

        suspicious_tlds = {"tk", "top", "xyz", "ru", "cc"}
        tld = value.rsplit(".", 1)[-1] if "." in value else ""
        if tld in suspicious_tlds:
            score += 10
            signals.append("위험 가능 TLD")

        verdict = "안전" if score < 10 else "의심" if score < 25 else "악성"
        return "\n".join([
            f"도메인: {value}",
            f"판정: {verdict}",
            f"위험 점수: {score}/100",
            f"감지 신호: {', '.join(signals) if signals else '없음'}",
        ])
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def ThreatIntelligenceSearchTool(query: str) -> str:
    """과거 피싱 사례 및 위협 인텔리전스 관련 키워드를 검색형으로 요약합니다.

    Args:
        query: 검색할 문자열 또는 IOC 관련 요약 정보입니다.

    Returns:
        위협 인텔리전스 요약 문자열입니다.
    """
    try:
        text = query.strip().lower()
        if not text:
            return "실패: 검색어가 비어 있습니다."

        keywords = [
            "phishing", "malware", "credential theft", "ioc", "known bad",
            "피싱", "악성", "자격 증명 탈취", "위협", "도메인", "url"
        ]
        hits = [kw for kw in keywords if kw in text]

        score = 0
        if hits:
            score += 30
        if any(term in text for term in ["urgent", "verify", "계정", "비밀번호", "결제"]):
            score += 10

        verdict = "안전" if score < 10 else "의심" if score < 25 else "악성"
        return "\n".join([
            f"검색어: {query}",
            f"판정: {verdict}",
            f"매칭 키워드: {', '.join(hits) if hits else '없음'}",
            f"참고 점수: {score}/100",
            "참고: 실제 외부 TI API 연동 전 기본 휴리스틱 결과입니다.",
        ])
    except Exception as e:
        return f"실패: {str(e)}"


@tool(parse_docstring=True)
def RiskScoreTool(
    email_text: str,
    urls: str | None = None,
    sender_email: str | None = None,
    sender_domain: str | None = None,
    attachment_info: str | None = None,
    historical_context: str | None = None,
) -> str:
    """이메일, URL, 발신자, 첨부파일, 과거 사례를 종합하여 위험도를 산정합니다.

    Args:
        email_text: 이메일 본문 및 헤더를 포함한 전체 텍스트입니다.
        urls: 추출되었거나 검사할 URL 목록 문자열입니다.
        sender_email: 발신자 이메일 주소입니다.
        sender_domain: 발신자 도메인입니다.
        attachment_info: 첨부파일 관련 정보입니다.
        historical_context: 과거 피싱 사례 또는 위협 인텔리전스 요약입니다.

    Returns:
        최종 판정과 점수가 포함된 문자열입니다.
    """
    try:
        text = email_text or ""
        lower_text = text.lower()
        score = 0
        signals = []

        if any(k in lower_text for k in ["긴급", "즉시", "verify", "urgent", "계정 정지", "비밀번호", "로그인"]):
            score += 15
            signals.append("긴급/인증 유도 문구")

        extracted_urls = []
        if urls:
            extracted_urls = [u.strip() for u in re.split(r"[\n,;\s]+", urls) if u.strip()]
        else:
            extracted_urls = re.findall(r'''https?://[^\s<>"]+|www\.[^\s<>"]+''', text, re.IGNORECASE)

        for u in extracted_urls:
            parsed = urlparse(u if u.startswith(("http://", "https://")) else f"http://{u}")
            host = (parsed.netloc or parsed.path).lower()
            if "@" in host:
                score += 20
                signals.append("URL 내 @ 포함")
            if len(host) > 30:
                score += 5
            if host.count("-") >= 3:
                score += 5
            if any(tld in host for tld in [".tk", ".top", ".xyz", ".ru"]):
                score += 10

        if sender_email:
            m = re.search(r"@([A-Za-z0-9.-]+\.[A-Za-z]{2,})$", sender_email.strip())
            if m:
                email_domain = m.group(1).lower()
                if sender_domain and sender_domain.lower() != email_domain:
                    score += 20
                    signals.append("발신자 도메인 불일치")
            else:
                score += 5
                signals.append("발신자 이메일 형식 비정상")

        if attachment_info:
            attachment_lower = attachment_info.lower()
            dangerous = [".exe", ".scr", ".bat", ".cmd", ".js", ".vbs", ".docm", ".xlsm", ".pptm"]
            if any(ext in attachment_lower for ext in dangerous):
                score += 25
                signals.append("위험 첨부파일 확장자")

        if historical_context:
            hist_lower = historical_context.lower()
            if any(k in hist_lower for k in ["phishing", "malware", "ioc", "악성", "피싱"]):
                score += 30
                signals.append("과거 위협 사례 연관")

        verdict = "안전" if score < 20 else "의심" if score < 50 else "악성"
        return "\n".join([
            f"최종 판정: {verdict}",
            f"위험 점수: {score}/100",
            f"감지 신호: {', '.join(signals) if signals else '없음'}",
            f"URL 수: {len(extracted_urls)}",
        ])
    except Exception as e:
        return f"실패: {str(e)}"


CUSTOM_TOOLS = [
    EmailParserTool,
    URLSecurityCheckTool,
    DomainLookupTool,
    ThreatIntelligenceSearchTool,
    RiskScoreTool,
]

email_parser_tool = EmailParserTool
url_security_check_tool = URLSecurityCheckTool
domain_lookup_tool = DomainLookupTool
threat_intelligence_search_tool = ThreatIntelligenceSearchTool
risk_score_tool = RiskScoreTool


도구 이름: domain_lookup_tool
도구 설명: 발신자 도메인의 기본 특성을 분석합니다.


## 5. 도구 정보 확인

생성된 도구의 메타데이터를 확인합니다.

In [ ]:
tool_function_name = CUSTOM_TOOLS[0]

print("=" * 80)
print("도구 정보")
print("=" * 80)
print(f"도구 이름: {tool_function_name.name}")
print(f"도구 설명: {tool_function_name.description}")
print(f"\n입력 스키마:")
print(tool_function_name.args_schema.schema())

## 6. 도구 단독 실행 테스트

**다양한 입력값으로 도구를 테스트하세요**

테스트 케이스를 최소 3개 이상 작성하세요:
1. 정상 케이스
2. 엣지 케이스 (경계값)
3. 에러 케이스 (잘못된 입력)


In [2]:
from tools import (
    EmailParserTool,
    URLSecurityCheckTool,
    DomainLookupTool,
    ThreatIntelligenceSearchTool,
    RiskScoreTool,
)

# 샘플 입력값 준비
sample_email_normal = '''From: attacker@example.com
To: victim@company.com
Subject: 긴급: 계정 인증 필요
Date: Mon, 1 Jan 2026 10:00:00 +0900

안녕하세요.
즉시 아래 링크에서 비밀번호를 재설정해 주세요.
https://example.com/login
'''

sample_email_edge = ''
sample_email_risk = '''From: service@security-update.xyz
To: user@company.com
Subject: 계정 정지 경고

첨부된 invoice.docm 파일을 열어 확인하세요.
http://bit.ly/abc123
'''

sample_url_normal = 'https://example.com/login'
sample_url_suspicious = 'http://bit.ly/abc123'
sample_domain_normal = 'company.com'
sample_domain_suspicious = 'security-update.xyz'
sample_query_normal = 'phishing campaign credential theft'
sample_query_edge = ''
sample_attachment_normal = 'invoice.docm'
sample_attachment_none = ''
sample_historical_normal = 'Known phishing IOC detected'
sample_historical_none = ''

print('테스트 1: EmailParserTool 정상 케이스')
print(EmailParserTool.invoke({'email_text': sample_email_normal}))
print()

print('테스트 2: EmailParserTool 엣지 케이스')
print(EmailParserTool.invoke({'email_text': sample_email_edge}))
print()

print('테스트 3: URLSecurityCheckTool 정상 케이스')
print(URLSecurityCheckTool.invoke({'url': sample_url_normal}))
print()

print('테스트 4: URLSecurityCheckTool 의심 케이스')
print(URLSecurityCheckTool.invoke({'url': sample_url_suspicious}))
print()

print('테스트 5: DomainLookupTool 정상 케이스')
print(DomainLookupTool.invoke({'domain': sample_domain_normal}))
print()

print('테스트 6: DomainLookupTool 의심 케이스')
print(DomainLookupTool.invoke({'domain': sample_domain_suspicious}))
print()

print('테스트 7: ThreatIntelligenceSearchTool 정상 케이스')
print(ThreatIntelligenceSearchTool.invoke({'query': sample_query_normal}))
print()

print('테스트 8: ThreatIntelligenceSearchTool 엣지 케이스')
print(ThreatIntelligenceSearchTool.invoke({'query': sample_query_edge}))
print()

print('테스트 9: RiskScoreTool 정상 케이스')
print(RiskScoreTool.invoke({
    'email_text': sample_email_risk,
    'urls': sample_url_suspicious,
    'sender_email': 'service@security-update.xyz',
    'sender_domain': 'security-update.xyz',
    'attachment_info': sample_attachment_normal,
    'historical_context': sample_historical_normal
}))
print()

print('테스트 10: RiskScoreTool 엣지 케이스')
print(RiskScoreTool.invoke({
    'email_text': '',
    'urls': '',
    'sender_email': '',
    'sender_domain': '',
    'attachment_info': sample_attachment_none,
    'historical_context': sample_historical_none
}))
print()


테스트 1: EmailParserTool 정상 케이스


NameError: name 'EmailParserTool' is not defined

## 7. 여러 도구 통합 테스트

팀에서 만든 여러 도구를 함께 테스트합니다.

**생성한 모든 도구를 리스트로 정리하세요**


In [ ]:
from tools import CUSTOM_TOOLS

print(f"총 {len(CUSTOM_TOOLS)}개의 도구가 준비되었습니다.\n")

for i, tool in enumerate(CUSTOM_TOOLS, 1):
    print(f"{i}. {tool.name}")
    print(f"   설명: {tool.description}")
    print()


## 프로젝트 체크리스트

**완료한 항목을 확인하세요:**

- [ ] 팀 도메인 선정 및 필요한 도구 정의 완료
- [ ] AI 프롬프트를 사용하여 도구 코드 생성 완료
- [ ] 최소 3개 이상의 도구 생성 완료
- [ ] 각 도구별 단독 실행 테스트 완료
- [ ] 정상/엣지/에러 케이스 테스트 완료
- [ ] 도구 메타데이터 확인 완료

---

## 다음 단계

생성한 도구를 domain-agent에 통합하세요:

1. `../src/domain-agent/tools.py` 파일 열기
2. TODO 주석을 참고하여 생성한 도구 코드 추가
3. `../src/domain-agent/agent.py` 파일 열기
4. TODO 주석을 참고하여 시스템 프롬프트와 도구 리스트 수정
5. LangGraph Studio로 테스트

---

## 참고 자료

- [LangChain Tools 문서](https://python.langchain.com/docs/modules/agents/tools/)
- [LangChain Custom Tools](https://python.langchain.com/docs/modules/agents/tools/custom_tools/)
- [@tool 데코레이터](https://python.langchain.com/docs/modules/agents/tools/custom_tools/#tool-decorator)